# M3L3 E16 — Memoria persistente con LangGraph (Resolution)
### Módulo 3 · Lecture 3 · Sistemas Multiagente

**Ejercicio paralelo:** E13 (sistema completo con estado y guardrails)

## ¿Qué vas a aprender hoy?
- usar `MemorySaver` para persistir estado entre turnos de conversación.
- aislar sesiones de distintos usuarios con `thread_id`.
- modelar guardrails como un nodo de validación explícito en el grafo.
- usar el LLM con historial de conversación para respuestas contextuales.


## ¿Qué necesitás saber antes?

Venís de E13 donde construiste `SessionState` como un dataclass que pasabas manualmente. En E16 el LLM genera respuestas usando el historial real.

> **MemorySaver:** checkpointer de LangGraph que persiste automáticamente el State después de cada `invoke`.

| E13 Python puro | E16 LangGraph |
|---|---|
| `state = SessionState()` manual | Primer `invoke` inicializa el thread |
| `response, state = handle_query(query, state)` | `app.invoke({"query": q}, config)` |
| Respuestas con keywords | LLM genera respuesta contextual con historial |
| `check_guardrails(state)` dentro de `handle_query` | Nodo `guardrail` visible en el diagrama |

```python
config_u1 = {"configurable": {"thread_id": "usuario_001"}}
app.invoke({"query": "vacaciones"}, config_u1)
app.invoke({"query": "¿y cómo?"},   config_u1)  # continúa la sesión
```


## Paso 1 — Elegí tu proveedor de LLM

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

In [ ]:
!pip install langgraph -q

import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

print("LangGraph listo.")

## Sección 1 — El State conversacional

> **State conversacional:** acumula información a lo largo de múltiples turnos. `Annotated[list, operator.add]` para el historial.


In [ ]:
knowledge_base = {
    "hr":      ["Vacaciones: 15 días hábiles por año.", "Licencias: pedido en PeopleOps."],
    "tech":    ["VPN: reiniciar cliente y validar MFA.", "Contraseña: restablecer en portal de identidad."],
    "billing": ["Facturas: cargar antes del día 25.", "Reembolsos: adjuntar recibo y centro de costo."],
}

KEYWORDS = {
    "hr":      ["vacaciones", "licencia", "beneficio", "seguro"],
    "tech":    ["vpn", "contraseña", "mfa", "notebook"],
    "billing": ["factura", "reembolso", "pago"],
}

MAX_TURNS = 5

def detect_domain(text: str, last_domain: str = "") -> str:
    text = text.lower()
    for d, words in KEYWORDS.items():
        if any(w in text for w in words):
            return d
    return last_domain or "unknown"

In [ ]:
class ConvState(TypedDict):
    query: str
    last_domain: str
    response: str
    turn_count: int
    blocked: bool
    history: Annotated[list[dict], operator.add]

## Sección 2 — El nodo guardrail

> **Guardrail:** nodo que bloquea si se superó el límite de turnos.

```
START → guardrail → (blocked=False) → respond → END
                 → (blocked=True)  → END
```

El nodo `respond` usa el LLM con historial para generar respuestas contextuales.

In [ ]:
def guardrail(state: ConvState) -> dict:
    if state["turn_count"] >= MAX_TURNS:
        return {"blocked": True, "response": "Límite de turnos alcanzado. Iniciá una nueva sesión."}
    return {"blocked": False, "turn_count": state["turn_count"] + 1}


def guardrail_router(state: ConvState) -> str:
    return END if state["blocked"] else "respond"


def respond(state: ConvState) -> dict:
    domain = detect_domain(state["query"], state.get("last_domain", ""))
    kb_context = "\n".join(knowledge_base.get(domain, [])) if domain != "unknown" else ""

    history_ctx = ""
    if state["history"]:
        recent = state["history"][-3:]
        history_ctx = "Historial reciente:\n" + "\n".join(
            f"- {h['query']} → {h['response']}" for h in recent
        )

    system = "Sos un asistente de soporte corporativo interno. Respondé brevemente en español."
    prompt = f"{system}\n\n{history_ctx}\n\nContexto:\n{kb_context}\n\nConsulta: {state['query']}"

    llm_response = llm.invoke(prompt)
    response_text = llm_response.content.strip()

    entry = {"turn": state["turn_count"], "query": state["query"], "response": response_text}
    return {
        "response": response_text,
        "last_domain": domain if domain != "unknown" else state.get("last_domain", ""),
        "history": [entry],
    }

## Sección 3 — Compilar con `MemorySaver`

```python
memory = MemorySaver()
app = graph.compile(checkpointer=memory)
```

In [ ]:
graph = StateGraph(ConvState)

graph.add_node("guardrail", guardrail)
graph.add_node("respond",   respond)

graph.add_edge(START, "guardrail")
graph.add_conditional_edges("guardrail", guardrail_router, {"respond": "respond", END: END})
graph.add_edge("respond", END)

memory = MemorySaver()
app = graph.compile(checkpointer=memory)
print("Grafo compilado con MemorySaver.")

In [ ]:
INITIAL = {"query": "", "last_domain": "", "response": "", "turn_count": 0, "blocked": False, "history": []}
SESSION = {"configurable": {"thread_id": "usuario_001"}}

turns = [
    "¿cuántos días de vacaciones tengo?",
    "¿y cómo las solicito?",
    "también tengo problemas con la VPN",
]

for i, query in enumerate(turns):
    state_in = {**INITIAL, "query": query} if i == 0 else {"query": query}
    result = app.invoke(state_in, SESSION)
    print(f"\nTurno {result['turn_count']}: {query}")
    print(f"Respuesta: {result['response']}")

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    init = {"query": "", "last_domain": "", "response": "", "turn_count": 0, "blocked": False, "history": []}
    cfg  = {"configurable": {"thread_id": "test_checks"}}

    r1 = app.invoke({**init, "query": "vacaciones"}, cfg)
    assert isinstance(r1["response"], str) and len(r1["response"]) > 5
    assert r1["turn_count"] == 1, f"esperaba turno 1: {r1['turn_count']}"

    r2 = app.invoke({"query": "¿y cómo las solicito?"}, cfg)
    assert r2["turn_count"] == 2, f"esperaba turno 2: {r2['turn_count']}"

    cfg2 = {"configurable": {"thread_id": "test_limit"}}
    r3 = app.invoke({**init, "turn_count": MAX_TURNS, "query": "algo"}, cfg2)
    assert r3["blocked"] == True, f"esperaba blocked=True: {r3['blocked']}"

    print("Checks E16 OK")

run_checks()

## ¿Qué aprendiste hoy?

- `MemorySaver` elimina el `state` manual de E13: LangGraph guarda y recupera el State con el `thread_id`.
- El LLM con historial genera respuestas contextuales: puede seguir el hilo de la conversación.
- Los guardrails como nodo explícito son auditables: aparecen en el diagrama.

---

## Resumen de la serie LangGraph completa

| Ejercicio | Concepto LangGraph | LLM | Paralelo Python puro |
|---|---|---|---|
| **E07** | Anatomía: State, Nodes, Edges, compile, invoke | `llm.invoke()` | — |
| **E08** | Router condicional | Clasificación semántica | E01 + E03 |
| **E09** | Fan-out con `Send` API | Detección múltiple de dominios | E04 |
| **E14** | Support bot como `StateGraph` | RAG + LLM en nodos | E11 |
| **E15** | Tool calling con `bind_tools()` | Function calling nativo | E12 |
| **E16** | Memoria con `MemorySaver` + guardrails | LLM con historial | E13 |
